# 🎬 Movie Recommendation System (Content-Based ML Project)

This project builds a simple movie recommendation system using machine learning techniques based on content similarity.

In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 1: Dataset loading and Exploration

We inspect the dataset to understand features such as movie titles, genres, and ratings.

In [6]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

movies.head()



,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
movies.shape, ratings.shape

((9125, 3), (100004, 4))

## Step 2: Feature Engineering

 combine movie metadata (genres and titles) to create meaningful features for similarity computation.

In [8]:
movies = movies[['movieId', 'title', 'genres']]
movies['genres'] = movies['genres'].fillna('')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Step 3: Recommendation Model

  TF-IDF vectorization and cosine similarity to measure how similar movies are to each other.

In [9]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [10]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [13]:
def recommend(title, movies, cosine_sim):
    # find index
    idx = movies[movies['title'] == title].index[0]
    
    # similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # top 10 (skip itself)
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies['title'].iloc[movie_indices]

recommend("Toy Story (1995)", movies, cosine_sim)

1815                                          Antz (1998)
2506                                   Toy Story 2 (1999)
3003       Adventures of Rocky and Bullwinkle, The (2000)
3217                     Emperor's New Groove, The (2000)
3805                                Monsters, Inc. (2001)
6705                               Shrek the Third (2007)
7146                       Tale of Despereaux, The (2008)
7945    Asterix and the Vikings (Astérix et les Viking...
8366                                         Turbo (2013)
8676                                Boxtrolls, The (2014)
Name: title, dtype: object

In [14]:
movies['combined_features'] = movies['genres'] + " " + movies['title']
movies['combined_features'] = movies['combined_features'].fillna('')

In [15]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['combined_features'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

## Step 4: Generate Recommendations

 I build a function that returns the top 10 most similar movies based on a selected movie title.

In [16]:
def recommend(title, movies, cosine_sim):
    if title not in movies['title'].values:
        return "Movie not found"

    idx = movies[movies['title'] == title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies.iloc[movie_indices][['title', 'genres']]

def recommend_with_scores(title, movies, cosine_sim):
    idx = movies[movies['title'] == title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top = sim_scores[1:11]

    results = []
    for i, score in top:
        results.append({
            "title": movies.iloc[i]['title'],
            "genres": movies.iloc[i]['genres'],
            "score": round(score, 3)
        })

    return pd.DataFrame(results)

recommend_with_scores("Toy Story (1995)", movies, cosine_sim)

,title,genres,score
0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.884
1,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.824
2,Toy Story of Terror (2013),Animation|Children|Comedy,0.675
3,"Toy, The (1982)",Comedy,0.529
4,We're Back! A Dinosaur's Story (1993),Adventure|Animation|Children|Fantasy,0.476
5,Now and Then (1995),Children|Drama,0.421
6,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy,0.397
7,Toy Soldiers (1991),Action|Drama,0.380
8,Up (2009),Adventure|Animation|Children|Drama,0.372
9,Balto (1995),Adventure|Animation|Children,0.369
